# Talker Comparison — PantoMatrix en Google Colab
Genera movimiento con **EMAGE**, **CaMN** o **DisCo** en la GPU de Colab, sin depender de ZeroGPU.

Está preparado para audios largos (incluidos ~2 minutos o más) procesándolos por chunks con solapamiento y fusionando las poses al final.

1. `Runtime > Change runtime type > GPU`.
2. Ejecutá las celdas en orden.
3. Subí un WAV/MP3.
4. Elegí modelo.
5. El notebook divide el audio automáticamente, genera cada tramo y descarga un JSON para Talker Comparison.


In [ ]:
!nvidia-smi
!rm -rf /content/PantoMatrix
!git clone --depth 1 https://github.com/PantoMatrix/PantoMatrix.git /content/PantoMatrix
%cd /content/PantoMatrix
!bash setup.sh

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, os, math, json, subprocess, numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly
import librosa, soundfile as sf

inp = Path('/content/talker_input')
chunks_dir = Path('/content/talker_chunks')
out = Path('/content/talker_output')
for p in (inp,chunks_dir,out):
    shutil.rmtree(p, ignore_errors=True); p.mkdir()
uploaded = files.upload()
src_name, src_bytes = next(iter(uploaded.items()))
src = inp/src_name; src.write_bytes(src_bytes)
print('Audio:', src)

In [ ]:
# Elegí: 'EMAGE', 'CaMN' o 'DisCo'
MODEL = 'EMAGE'
CHUNK_SEC = 8.0
OVERLAP_SEC = 1.0
TARGET_SR = 16000

audio, sr = librosa.load(str(src), sr=TARGET_SR, mono=True)
duration = len(audio)/TARGET_SR
segments=[]; start=0.0; idx=0
while start < duration-1e-6:
    end=min(duration,start+CHUNK_SEC)
    a=int(start*TARGET_SR); b=int(end*TARGET_SR)
    chunk=audio[a:b]
    target_len=int(CHUNK_SEC*TARGET_SR)
    if len(chunk)<target_len:
        chunk=np.pad(chunk,(0,target_len-len(chunk)))
    p=chunks_dir/f'chunk_{idx:03d}.wav'
    sf.write(p,chunk,TARGET_SR)
    segments.append((start,end,p))
    if end>=duration: break
    start=end-OVERLAP_SEC; idx+=1
print(f'Duración: {duration:.1f}s · chunks: {len(segments)}')

In [ ]:
import pathlib, shutil
py = '/content/py39/bin/python' if pathlib.Path('/content/py39/bin/python').exists() else 'python'
script_map={'EMAGE':'test_emage_audio.py','CAMN':'test_camn_audio.py','DISCO':'test_disco_audio.py'}
script=script_map[MODEL.upper()]
chunk_npzs=[]
for i,(s,e,p) in enumerate(segments):
    work=Path('/content/current_chunk'); shutil.rmtree(work,ignore_errors=True); work.mkdir()
    shutil.copy2(p,work/p.name)
    save=out/f'chunk_{i:03d}'; save.mkdir(parents=True,exist_ok=True)
    cmd=[py,script,'--audio_folder',str(work),'--save_folder',str(save)]
    print(f'[{i+1}/{len(segments)}] {s:.1f}-{e:.1f}s')
    subprocess.run(cmd,cwd='/content/PantoMatrix',check=True)
    found=list(save.rglob('*.npz'))
    if not found:
        found=list(Path('/content/PantoMatrix').rglob(f'{p.stem}*output*.npz'))
    if not found: raise RuntimeError(f'No encontré NPZ para chunk {i}')
    chunk_npzs.append(max(found,key=lambda x:x.stat().st_mtime))
print('NPZ generados:',len(chunk_npzs))

In [ ]:
SMPLX={'hips':0,'leftUpperLeg':1,'rightUpperLeg':2,'spine':3,'leftLowerLeg':4,'rightLowerLeg':5,'chest':9,'leftFoot':10,'rightFoot':11,'neck':12,'leftShoulder':13,'rightShoulder':14,'head':15,'leftUpperArm':16,'rightUpperArm':17,'leftForeArm':18,'rightForeArm':19,'leftHand':20,'rightHand':21}
def aa_quat(v):
    x,y,z=map(float,v); a=math.sqrt(x*x+y*y+z*z)
    if a<1e-8:return [0,0,0,1]
    s=math.sin(a/2)/a; return [x*s,y*s,z*s,math.cos(a/2)]
def slerp(a,b,t):
    a=np.array(a,float); b=np.array(b,float); dot=float(np.dot(a,b))
    if dot<0: b=-b; dot=-dot
    if dot>0.9995:
        q=a+(b-a)*t; q/=np.linalg.norm(q); return q.tolist()
    th=math.acos(max(-1,min(1,dot))); sn=math.sin(th)
    return (math.sin((1-t)*th)/sn*a + math.sin(t*th)/sn*b).tolist()
def npz_frames(path):
    d=np.load(path,allow_pickle=True)
    poses=d['poses'] if 'poses' in d else d['motion']
    poses=np.asarray(poses)
    if poses.ndim>2: poses=poses.reshape(poses.shape[0],-1)
    frames=[]
    for row in poses:
        joints={}
        for name,j in SMPLX.items():
            joints[name]={'rotation':[0,0,0,1] if name=='hips' else aa_quat(row[j*3:j*3+3])}
        frames.append({'root':{'position':[0,0,0]},'joints':joints})
    return frames
parts=[npz_frames(p) for p in chunk_npzs]
fps=30; overlap_frames=int(OVERLAP_SEC*fps)
merged=list(parts[0])
for part in parts[1:]:
    n=min(overlap_frames,len(merged),len(part))
    for i in range(n):
        t=(i+1)/(n+1); A=merged[-n+i]; B=part[i]
        for j in SMPLX:
            qa=A['joints'][j]['rotation']; qb=B['joints'][j]['rotation']
            A['joints'][j]['rotation']=slerp(qa,qb,t)
    merged.extend(part[n:])
target_frames=int(round(duration*fps))
merged=merged[:target_frames]
model_id={'EMAGE':'pantomatrix','CAMN':'camn','DISCO':'disco'}[MODEL.upper()]
result={'model':model_id,'generator':MODEL+' via Google Colab chunked','fps':fps,'source_duration_sec':duration,'chunk_sec':CHUNK_SEC,'overlap_sec':OVERLAP_SEC,'frames':merged}
json_path=Path('/content')/f'{MODEL.lower()}-colab-long-motion.json'
json_path.write_text(json.dumps(result))
print('Frames:',len(merged),'duración final:',len(merged)/fps,'s')
files.download(str(json_path))